In [ ]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import warpSPHCore_config as swc
from typing import Any
swc.configure(precision="float64", dim=Any) # precision: float16|half|float32|single|float64|double

import warpSPHCore as sph
from warpSPHCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from warpSPHIntegrators.integration import *
from warpSPHCore import *

# This library
from warpSPH import *
from warpSPH.modules.timestep.compressible import computeTimestep

# The case utilities that contain all the case setup functions for the various test cases
from warpSPH.caseUtils import *

# Linear Wave

This notebook runs the 1D Linear Wave benchmark in the compressible SPH suite.

The case initializes a small-amplitude acoustic perturbation and tracks its propagation to evaluate dispersion, phase accuracy, and numerical dissipation.

This notebook follows the same reusable structure used across all 15 compressible benchmark cases:

1. Configure imports and numeric precision.
2. Define case-specific physical parameters and initial-condition data.
3. Build domain, solver, and scheme configuration from shared builders.
4. Sample and initialize particles/state for the selected case.
5. Run the time integration loop with diagnostics and adaptive timestep control.
6. Export trajectory/state snapshots and generate image frames during the run.
7. Finalize outputs by writing final state data and rendering media artifacts (for example MP4/GIF).

Precision note: switching between single and double precision is controlled in the import/configuration block. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/02-Linear_wave.gif)

In [ ]:
E0 = 1
nx = 200

A = 1e-6
lamda = 1
rho0 = 1.0
c_s = 1
gamma = 5/3


initialStateDict = {
    'rho0': rho0,
    'E0': E0,
    'gamma': gamma,
    'nx': nx,
    'A': A,
    'lambda': lamda,
    'c_s': c_s,
}

In [ ]:
L = 1
dim = 1
n_h = 4
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
dtype = get_torch_precision()

config, integrator = buildConfig(
    domain = buildDomainDescription(L, dim, True, device, dtype),
    dim = dim,
    kernel = KernelFunctions.Wendland4,
    targetNeighbors = n_h_to_nH(4, dim),
    supportMode = SupportScheme.Gather,
    gradientMode = GradientScheme.Difference,
    laplacianMode = LaplacianScheme.Brookshaw,
    integrationScheme = IntegrationSchemeType.rungeKutta2,
    samplingScheme = SamplingScheme.regular,
    device = device,
    dtype = dtype,
    dt = 1e-3,
    adaptiveDt = True,
    cflFactor=0.3,
)
config.nx = nx


scheme = CompressibleSPHScheme.CRKSPH
bundle = buildScheme(scheme)
SimulationSystem, SimulationState = bundle.SimulationSystem, bundle.SimulationState
SimulationUpdate = bundle.SimulationUpdate
fn, export_fn, import_fn = bundle.stepFunction, bundle.exportFunction, bundle.importFunction

compressibleSPHConfig = bundle.SimulationConfig(
    gamma = gamma,
    rho0 = rho0,
)
integrator = getIntegrator(config.integrationScheme)

compressibleSPHConfig.viscositySwitchParams.scheme = ViscositySwitch.NoneSwitch
compressibleSPHConfig.adaptiveSupportScheme = AdaptiveSupportScheme.Owen

In [ ]:
compressibleSystem = sampleLinearWave(
    nx,
    config,
    compressibleSPHConfig,
    SimulationState,
    SimulationSystem,
    A, lamda, c_s, rho0, gamma,
    nIters = 16,
    supportScheme = AdaptiveSupportScheme.Owen,
)

In [ ]:
print(f"Initial timestep: {config.dt}")

c_mean = compressibleSystem.state.soundspeeds.mean().cpu().item()
c_max = compressibleSystem.state.soundspeeds.max().cpu().item()
c_min = compressibleSystem.state.soundspeeds.min().cpu().item()
timeLimit = 1.0/c_mean
timesteps = int(timeLimit / config.dt)

print(timesteps, config.dt)

runningState = compressibleSystem.initializeNewState()

In [ ]:
fig, axis = plt.subplots(1, 3, figsize = (10, 5), squeeze=False)

plotState(fig, axis, runningState, compressibleSystem, config, compressibleSPHConfig, rho0, A, lamda, c_s)
fig.tight_layout()


In [ ]:
caseName = '02-linearWave'

kineticEnergy_ = 0.5 * (torch.linalg.norm(compressibleSystem.state.velocities, dim = -1) **2 * compressibleSystem.state.masses).sum()
thermalEnergy_ = (compressibleSystem.state.internalEnergies * compressibleSystem.state.masses).sum()
totalEnergy = kineticEnergy_ + thermalEnergy_
exportPath = prepExport(f'{caseName}', config, compressibleSPHConfig, scheme, export_fn)
exportSimulationSystem(exportPath, 'initialState', scheme, compressibleSystem, exportAdjacency = False, stages = None, exportStagesAdjacency = False, extraData = dict({
    'kineticEnergy': kineticEnergy_,
    'thermalEnergy': thermalEnergy_,
    'totalEnergy': totalEnergy,
    'frame_num': 0,
}, **initialStateDict))

imagePath = f'{exportPath}/images'
os.makedirs(imagePath, exist_ok = True)
fig.savefig(f'{imagePath}/frame_{0:05d}.png')

In [ ]:
# config.dt = 1e-4
timeLimit = 1 / c_min
t_limit = timeLimit
config.dt = t_limit / 1000
nSteps = int(t_limit / config.dt) 

print(f"Running with dt: {config.dt}, which gives nSteps: {nSteps}")
# nSteps = 20

runningState = compressibleSystem.initializeNewState()

trajectory = []

priorStep = None
for i in (tq := tqdm(range(nSteps), leave = True)):
    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    result = integrator.function(
        state = runningState,
        f = fn,
        dt = config.dt,  
        config = config,
        schemeConfig = compressibleSPHConfig,
        verbose = False,
        # priorStep = priorStep
    )
    end.record()
    torch.cuda.synchronize()
    priorStep = result.stages[-1]
    timing = begin.elapsed_time(end)

    runningState = result.state
    kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
    thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
    totalEnergy = kineticEnergy + thermalEnergy

    trajectory.append(
        (i, (i+1)*config.dt, totalEnergy.item(), kineticEnergy.item(), thermalEnergy.item(), timing)
,     )

    tq.set_description(f"Step {i+1}/{nSteps}, time: {(i+1)*config.dt:8.4g}/{t_limit:8.4g}, TE: {totalEnergy:.3g}, KE: {kineticEnergy:.3g}, IE: {thermalEnergy:.3g}")
    # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
    # break
    if i % 10 == 0 or i == nSteps - 1:
        plotState(fig, axis, runningState, compressibleSystem, config, compressibleSPHConfig, rho0, A, lamda, c_s)
        # fig.tight_layout()
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(f'{imagePath}/frame_{i:05d}.png')

    if i % 50 == 0:
        exportSimulationSystem(exportPath, f'state_{i:04d}', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**initialStateDict, **{
            'kineticEnergy': kineticEnergy_,
            'thermalEnergy': thermalEnergy_,
            'totalEnergy': totalEnergy,
            'frame_num': i,
        }))


exportSimulationSystem(exportPath, f'finalState', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**initialStateDict, **{
    'kineticEnergy': kineticEnergy_,
    'thermalEnergy': thermalEnergy_,
    'totalEnergy': totalEnergy,
    'frame_num': i,
}))

In [ ]:

ffmpeg_cmd = "ffmpeg -y -loglevel error -hide_banner -framerate 50 -f image2 -pattern_type glob -i 'frame_*.png' -c:v libx264 -pix_fmt yuv420p -b:v 10M output.mp4"
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4  -vf "fps=50,scale=540:-1:flags=lanczos,palettegen" palette.png'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4 -i palette.png -filter_complex "fps=25,scale=540:-1:flags=lanczos[x];[x][1:v]paletteuse" out.gif'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)

# now copy the output.mp4 and out.gif to the parent directory for easier access
import shutil
shutil.copy(f'{imagePath}/output.mp4', f'{exportPath}/output.mp4')
shutil.copy(f'{imagePath}/out.gif', f'{exportPath}/out.gif');